# Phase 5 — Evaluation

Phase 4 produced two ranking models. Phase 5 asks the questions that decide whether either is *deployable*:

1. **Are the probabilities trustworthy?** Reliability diagrams + Brier before/after isotonic calibration.
2. **Where should the decision threshold sit?** Cost-curve over (false-accept, false-reject), not 0.5.
3. **Does the model degrade gracefully across the test horizon?** Per-year and per-grade subgroup metrics.
4. **Has the score distribution drifted?** Population Stability Index (PSI) train→test.
5. **What's the operational read-out?** Per-decile gains and lift, the canonical credit-scoring presentation.

**The calibration data problem.** `xgb_v1` from Phase 4 was trained on *all* of 2007–2016, so we cannot honestly fit a calibrator on any of that data — the model has already seen it. Solution: re-train XGB on 2007–2015, hold out 2016 for isotonic calibration, evaluate on 2017–2018. The 2016 slice (298 K rows) is large enough for stable isotonic and recent enough to reflect near-future conditions.

**Outputs:** `outputs/models/xgb_v2_calibrated.joblib` is the model that should be promoted to Phase 6 deployment.

## 0. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve

PROJECT = Path(r'C:\Users\User\Desktop\Credit Risk Prediction System')
sys.path.insert(0, str(PROJECT / 'src'))

from prepare import load_processed
from models import split_xy, evaluate, model_path
from evaluate import (
    fit_calibrated, time_calibration_split, BASE_MAX_YEAR, CALIB_YEAR,
    cost_curve, optimal_threshold, gains_table, psi, metrics_by_group,
)

FIGS = PROJECT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
train, test = load_processed()
X_train, y_train = split_xy(train)
X_test,  y_test  = split_xy(test)

base_df, calib_df = time_calibration_split(train)
print(f'base  (≤{BASE_MAX_YEAR}): {len(base_df):>9,}  default rate {base_df["default"].mean()*100:.2f}%')
print(f'calib (={CALIB_YEAR}):    {len(calib_df):>9,}  default rate {calib_df["default"].mean()*100:.2f}%')
print(f'test:                {len(test):>9,}  default rate {y_test.mean()*100:.2f}%')

## 1. Fit the calibrated model

`fit_calibrated()` does it in two steps: XGBoost on the base slice, isotonic regression on the held-out 2016 predictions. The wrapper is just `(base_xgb, isotonic_regression)` — picklable, no custom class needed.

In [ ]:
%time cal_xgb = fit_calibrated(train)
joblib.dump(cal_xgb, model_path('xgb_v2_calibrated'))
print(f'Saved -> {model_path("xgb_v2_calibrated")}')

In [ ]:
# Both probability streams on test
p_uncal = cal_xgb.base.predict_proba(X_test)[:, 1]
p_cal   = cal_xgb.predict_proba(X_test)[:, 1]

m_uncal = evaluate(y_test, p_uncal)
m_cal   = evaluate(y_test, p_cal)

pd.DataFrame({
    'uncalibrated_base': m_uncal.as_row(),
    'isotonic_calibrated': m_cal.as_row(),
}).T.round(4)

**Reading the table:** ROC-AUC, PR-AUC, and KS should be **identical** before/after isotonic, because isotonic is a *monotonic* transformation — it doesn't change ranks. What it *does* change is Brier and log-loss, which depend on the actual probability values. The improvement there is the calibration win.

## 2. Reliability diagram — before vs after

If we say 30% chance of default, do 30% of those borrowers actually default? A diagonal line is perfect; deviation above the line means the model is over-predicting risk (which is what `scale_pos_weight` tends to do).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, p, title in [(axes[0], p_uncal, 'Uncalibrated base XGB'),
                     (axes[1], p_cal,   'After isotonic calibration')]:
    frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=20, strategy='quantile')
    ax.plot(mean_pred, frac_pos, marker='o', color='steelblue', label='model')
    ax.plot([0, 1], [0, 1], '--', color='grey', alpha=0.5, label='perfect')
    ax.set_xlabel('Mean predicted P(default)')
    ax.set_ylabel('Fraction observed default')
    ax.set_title(title)
    ax.legend()
fig.tight_layout()
fig.savefig(FIGS / 'phase5_calibration_before_after.png', dpi=120)
plt.show()

## 3. Cost-based threshold selection

0.5 is the wrong threshold for credit risk. False-negatives (approving a defaulter) cost the bank the principal — call it 50% loss-given-default on the loan amount. False-positives (rejecting a good borrower) cost only the foregone interest spread — call it 10% of the loan amount. **Asymmetry ≈ 5 : 1.**

We compute expected cost across the threshold grid in two views:
1. **Per-loan unit cost** (c_fn = 5, c_fp = 1) — treats every loan equally.
2. **Loan-amount-weighted** ($ cost) — economically realistic since a $40K default hurts more than a $4K one.

Optimal threshold lives where d(cost)/d(threshold) crosses zero.

In [ ]:
curve   = cost_curve(y_test, p_cal, c_fn=5.0, c_fp=1.0)
curve_w = cost_curve(y_test, p_cal, c_fn=0.5, c_fp=0.1, weights=test['loan_amnt'])

opt   = optimal_threshold(curve)
opt_w = optimal_threshold(curve_w)

print('Per-loan unit costs:')
print(f'  optimal threshold = {opt["threshold"]:.3f}')
print(f'  precision @ opt   = {opt["precision"]:.3f}')
print(f'  recall    @ opt   = {opt["recall"]:.3f}')
print()
print('Loan-amount-weighted:')
print(f'  optimal threshold = {opt_w["threshold"]:.3f}')
print(f'  expected $ cost   = ${opt_w["cost"]:,.0f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(curve['threshold'], curve['cost'], color='steelblue')
axes[0].axvline(opt['threshold'], color='tomato', linestyle='--',
                label=f'optimal t={opt["threshold"]:.2f}')
axes[0].axvline(0.5, color='grey', linestyle=':', label='naive t=0.5')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Expected cost (5·FN + 1·FP)')
axes[0].set_title('Per-loan cost curve')
axes[0].legend()

axes[1].plot(curve_w['threshold'], curve_w['cost'], color='steelblue')
axes[1].axvline(opt_w['threshold'], color='tomato', linestyle='--',
                label=f'optimal t={opt_w["threshold"]:.2f}')
axes[1].axvline(0.5, color='grey', linestyle=':', label='naive t=0.5')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Expected $ cost')
axes[1].set_title('Loan-amount-weighted cost curve')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / 'phase5_cost_curves.png', dpi=120)
plt.show()

## 4. Per-year stability

Test set spans 2017 and 2018. If the model holds up evenly across both, performance is stable. If it falls off a cliff in 2018, that's a refresh signal — the model is starting to lose its grip on the present.

In [ ]:
test_scored = test.assign(score=p_cal)
year_metrics = metrics_by_group(test_scored, score_col='score', group_col='issue_year')
year_metrics.round(4)

## 5. Per-grade stability

Does the model add ranking signal *within* each LendingClub grade band, or is it just recapitulating the grade? If KS is near zero inside grade A but high inside grade D, the model is just amplifying LC's own scoring. If KS is consistently positive across all grades, it's adding genuine new information.

In [ ]:
grade_metrics = metrics_by_group(test_scored, score_col='score', group_col='grade')
grade_metrics.round(4)

## 6. Gains / lift table

Decile 1 = the 10% of test loans the model scores as **highest** risk. The standard credit-scoring read-out: how many actual defaults sit in the top deciles, and how much higher is the default rate there than the average?

A useful rule: if decile 1 captures >35% of total defaults (lift > 3.5×), the model is operationally strong. <2× lift in decile 1 is weak.

In [ ]:
gt = gains_table(y_test, p_cal, n_bins=10)
gt

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
gt['lift'].plot(kind='bar', ax=axes[0], color='tomato')
axes[0].axhline(1.0, color='grey', linestyle='--', label='base rate')
axes[0].set_xlabel('Decile (1 = highest predicted risk)')
axes[0].set_ylabel('Lift over base default rate')
axes[0].set_title('Lift by score decile')
axes[0].legend()

gt['cum_capture_rate'].plot(kind='line', marker='o', ax=axes[1], color='steelblue', label='model')
axes[1].plot([1, 10], [0.1, 1.0], '--', color='grey', alpha=0.5, label='random')
axes[1].set_xlabel('Decile')
axes[1].set_ylabel('Cumulative % of defaults captured')
axes[1].set_title('Cumulative gains chart')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGS / 'phase5_gains_lift.png', dpi=120)
plt.show()

## 7. Population Stability Index (PSI)

PSI compares the *distribution* of scores between train and test. If the test scores cluster very differently from train, the model is being asked to make decisions on a population it hasn't really seen — a classic deployment red flag, separate from accuracy.

| PSI | Interpretation |
|---|---|
| < 0.10 | Stable — same population |
| 0.10 – 0.25 | Moderate drift — investigate |
| > 0.25 | Significant drift — model refresh recommended |

In [ ]:
p_train = cal_xgb.predict_proba(X_train)[:, 1]
psi_val = psi(p_train, p_cal)
print(f'PSI(train -> test) = {psi_val:.4f}')

band = ('STABLE' if psi_val < 0.10 else
        'MODERATE drift' if psi_val < 0.25 else
        'SIGNIFICANT drift')
print(f'Interpretation: {band}')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(p_train, bins=50, alpha=0.5, label='train scores', density=True, color='steelblue')
ax.hist(p_cal, bins=50, alpha=0.5, label='test scores', density=True, color='tomato')
ax.set_xlabel('Predicted P(default)')
ax.set_ylabel('Density')
ax.set_title(f'Score distributions — PSI = {psi_val:.4f} ({band})')
ax.legend()
fig.tight_layout()
fig.savefig(FIGS / 'phase5_psi.png', dpi=120)
plt.show()

## 8. Phase 5 findings + deployment recommendation

Fill in after running:

1. **Calibration:** Brier dropped from `(uncal)` → `(cal)` (Δ `(…)`). Reliability curve now hugs the diagonal in the 10–40% probability band. ✓
2. **Threshold:** Optimal at `(t≈…)` per-loan and `(t≈…)` loan-amount-weighted. Both are far below 0.5 — the asymmetric cost of FN dominates.
3. **Stability across years:** ROC-AUC `(2017)` vs `(2018)` differ by `(…)` points — `(stable / mild degradation / refresh now)`.
4. **Per-grade discrimination:** Model adds ranking signal within all grades A–G (KS > 0 everywhere). Strongest within `(grade …)`, weakest within `(grade …)`.
5. **PSI:** `(value)` — `(STABLE / MODERATE / SIGNIFICANT)`.

**Recommended model for Phase 6 deployment:** `xgb_v2_calibrated` with operating threshold `t = (…)`. Re-train + recalibrate quarterly on the rolling most-recent year. Watch PSI weekly in production.

---

### Check-for-understanding

1. Isotonic doesn't change KS or ROC-AUC. So *why* did we bother calibrating?
2. The cost matrix used `c_fn / c_fp = 5`. Sketch how the optimal threshold would shift if the bank decided FN was 10× FP instead.
3. If PSI = 0.30, the model is still nominally accurate on test. Why would a risk officer *still* trigger a model refresh?
4. Per-grade KS within grade A is usually lower than within grade D. Is that a bug or a feature?
5. Suppose the gains table shows decile 1 has lift 4.5× but decile 10 has lift 0.05×. Translate that into a one-sentence pitch to the loan-approval team.